# Challenge 03 - SolarPark (predict ahead)

Regression of the power produced by a photovoltaic plant during the next hour (t+1), using only the features of the row at time t. The goal is to minimize the RMSE on the test set. The predictions are saved in `Es5Predictions.txt`.

## Setup

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import (ExtraTreesRegressor, HistGradientBoostingRegressor,
                              RandomForestRegressor, VotingRegressor)
from sklearn.model_selection import TimeSeriesSplit

# Configuration
db_path = '../../DBs/SolarPark'
train_file = 'train.txt'
test_file = 'test.txt'
result_path = '../Es5Predictions.txt'

FEATURES = ['hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'month_sin', 'month_cos',
            'Ta', 'Tm', 'I3', 'I15', 'dT', 'I_mean']

## Helper Functions

CSV loading, feature engineering (cyclical encoding of the date, temperature difference `dT` and mean irradiance `I_mean`), construction of the predict-ahead pairs, temporal cross-validation and the export of the predictions.

In [2]:
def load_dataset(db_path, filename):
    return pd.read_csv(os.path.join(db_path, filename))


def cyclical(values, period):
    angle = 2 * np.pi * values / period
    return np.sin(angle), np.cos(angle)


def build_features(df):
    """Features computed from the row at time t only (no past rows, no P)."""
    date = pd.to_datetime(df['Date'], format='%m/%d/%y %I:%M %p')
    x = pd.DataFrame(index=df.index)
    x['hour_sin'], x['hour_cos'] = cyclical(df['Time Frame'], 24)
    x['doy_sin'], x['doy_cos'] = cyclical(date.dt.dayofyear, 365.25)
    x['month_sin'], x['month_cos'] = cyclical(date.dt.month, 12)
    x['Ta'] = df['Ta (C)']
    x['Tm'] = df['Tm (C)']
    x['I3'] = df['I3 (W/m2)']
    x['I15'] = df['I15 (W/m2)']
    x['dT'] = x['Tm'] - x['Ta']
    x['I_mean'] = (x['I3'] + x['I15']) / 2
    return x[FEATURES]


def make_predict_ahead(df, max_gap=1.5, min_ratio=0.4, min_irradiance=200):
    """Pair the features of row t with P of row t+1.

    A pair is kept when the next row is the next hour of the day (Time Frame) and it is not a
    jump over missing data (date at most `max_gap` hours later). Night rows (Time Frame 22-2)
    are zero-padded copies with a frozen date: pairs between them have gap 0 and are kept
    (target 0), as is the jump from the last padded row (2) to the first real one (3). They
    matter because the test set contains night rows too.

    Returns x, y and `anomalous`, a mask of pairs whose target row looks like a plant outage
    (high irradiance at t+1 but P(t+1) < `min_ratio` * I15(t+1)); they should be removed from
    the training set only (cleaning), never used as input.
    """
    date = pd.to_datetime(df['Date'], format='%m/%d/%y %I:%M %p')
    gap_hours = (date.shift(-1) - date).dt.total_seconds() / 3600
    next_hour = df['Time Frame'].shift(-1) == df['Time Frame'] % 24 + 1
    valid = (next_hour & (gap_hours.between(0, max_gap) | (df['Time Frame'] == 2))).to_numpy()

    target = df['P (kW)'].shift(-1)[valid].to_numpy()
    next_irr = df['I15 (W/m2)'].shift(-1)[valid].to_numpy()
    anomalous = (next_irr > min_irradiance) & (target < min_ratio * next_irr)
    return build_features(df)[valid].reset_index(drop=True), pd.Series(target), anomalous


def rmse_cv(model, x, y, train_mask=None, n_splits=5):
    """RMSE of each TimeSeriesSplit fold (ordered temporal blocks), as an array.

    `train_mask` excludes patterns (e.g. anomalies) from the training folds only, so every
    validation fold is still scored on all its patterns, like the real test set.
    """
    x, y = np.asarray(x), np.asarray(y)
    train_mask = np.ones(len(y), bool) if train_mask is None else np.asarray(train_mask)
    scores = []
    for train_idx, val_idx in TimeSeriesSplit(n_splits).split(x):
        train_idx = train_idx[train_mask[train_idx]]
        fold_model = clone(model).fit(x[train_idx], y[train_idx])
        pred = np.clip(fold_model.predict(x[val_idx]), 0, None)
        scores.append(np.sqrt(np.mean((pred - y[val_idx]) ** 2)))
    return np.array(scores)


def save_predictions(predictions, path):
    """One value per line (np.savetxt default format '%.18e'), clipped to >= 0."""
    np.savetxt(path, np.clip(predictions, 0, None))
    print(f'Predictions successfully saved to {path}')

## Data Loading

The features of row t are paired with the power of row t+1. Training pairs whose target row looks like a plant outage (high irradiance but very low power) are flagged as anomalous: they are removed from the training set only.

In [3]:
train_df = load_dataset(db_path, train_file)
test_df = load_dataset(db_path, test_file)
train_x, train_y, anomalous = make_predict_ahead(train_df)
test_x = build_features(test_df)
print('Training pairs: %d (%d anomalous removed), test rows: %d' % (len(train_x), anomalous.sum(), len(test_x)))

Training pairs: 7564 (68 anomalous removed), test rows: 7900
